# Extract WordNet Meronym & Holonym trees

Loops over **every** synset in Open English WordNet (`oewn:2025`) and, for the
synsets that have a **meronym** or **holonym** relationship, writes an indented
tree to `oewn2024_meronym_holonym.txt`.

Line format (same as the reference `oewn2024_i15.docx`, plus indentation):

```
(level) {lemma1, lemma2} [synset-id]: definition
```

Each level deeper (a directly-related synset) is indented by one extra `\t` (tab).
The output is split into two sections: **MERONYM** and **HOLONYM**.


In [ ]:
import os
import wn

# The notebook lives in <repo>/Ms_Trang, so the repo root is one level up.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Use a dedicated data directory for oewn:2025 so we don't touch any existing
# wn.db that may have been built with a different wn schema version.
DATA_DIR = os.path.join(REPO_ROOT, "data", "wn2025")
os.makedirs(DATA_DIR, exist_ok=True)
wn.config.data_directory = DATA_DIR

LEXICON = "oewn:2025"          # Open English WordNet 2025
OUTPUT = "oewn2024_meronym_holonym.txt"

# Install the lexicon once (no-op if already present).
#   Behind a corporate proxy, wn.download() may fail SSL verification. In that
#   case fetch the file manually and register it from disk with wn.add():
#     curl -fsSL -o ../data/english-wordnet-2025.xml.gz \
#          https://en-word.net/static/english-wordnet-2025.xml.gz
LOCAL_XML = os.path.join(REPO_ROOT, "data", "english-wordnet-2025.xml.gz")
if not wn.lexicons(lexicon=LEXICON):
    if os.path.exists(LOCAL_XML):
        wn.add(LOCAL_XML)
    else:
        wn.download(LEXICON)

en = wn.Wordnet(LEXICON)
print("Loaded", LEXICON, "-", len(en.synsets()), "synsets")


In [ ]:
# All meronym / holonym sub-relations defined in the WordNet schema.
MERONYM_RELATIONS = ["mero_part", "mero_member", "mero_substance",
                     "mero_location", "mero_portion"]
HOLONYM_RELATIONS = ["holo_part", "holo_member", "holo_substance",
                     "holo_location", "holo_portion"]


def format_line(synset, level):
    """One line matching the reference docx, prefixed with `level` tabs."""
    indent = "\t" * level
    lemmas = ", ".join(synset.lemmas())
    definition = synset.definition() or ""
    return f"{indent}({level}) {{{lemmas}}} [{synset.id}]: {definition}"


def related(synset, relations):
    """Directly-related synsets across all the given relation types (deduped)."""
    out, seen = [], set()
    for rel in relations:
        for tgt in synset.get_related(rel):
            if tgt.id not in seen:
                seen.add(tgt.id)
                out.append(tgt)
    return out


def write_tree(synset, relations, level, out, path):
    """Recursively write a synset and its related synsets as an indented tree.

    `path` guards against cycles (e.g. part/whole loops) along the current
    branch: a synset already on the path is not expanded again.
    """
    out.write(format_line(synset, level) + "\n")
    path.add(synset.id)
    for child in related(synset, relations):
        if child.id not in path:
            write_tree(child, relations, level + 1, out, path)
    path.discard(synset.id)


In [ ]:
# Build the two sections and write the file.
all_synsets = en.synsets()

with open(OUTPUT, "w", encoding="utf-8") as out:
    for title, relations in (("MERONYM", MERONYM_RELATIONS),
                             ("HOLONYM", HOLONYM_RELATIONS)):
        out.write(f"===== {title} =====\n\n")
        roots = 0
        for synset in all_synsets:
            if related(synset, relations):          # only synsets with the relation
                write_tree(synset, relations, 0, out, set())
                out.write("\n")
                roots += 1
        out.write("\n")
        print(f"{title}: wrote {roots} root synsets")

print("Done ->", os.path.abspath(OUTPUT))
